# Experiment 1: Pause after a block, change its output, resume

GPT-2 small is made of **12 blocks** stacked one after another (numbered 0 to 11).
Each block takes a table of numbers (768 numbers for every word-piece) and passes an updated table to the next block.

In this notebook we:
1. Run the normal model and note its prediction.
2. Run the model **one block at a time** ourselves, and check that the answer is exactly the same.
3. **PAUSE** after a chosen block and **save** its output.
4. **MODIFY** the saved output.
5. **RESUME** from the next block, **without rerunning** the earlier blocks.
6. Compare the normal and modified predictions.

## Step 0: Install (Colab only)
Colab already has PyTorch and Transformers. This notebook was tested with PyTorch 2.14.1 and Transformers 5.20.2.
If your versions differ and something breaks, uncomment the line below and run it.

In [1]:
# !pip install -q transformers==5.20.2

## Step 1: Load GPT-2 small
`torch.set_grad_enabled(False)` turns off training features, since we only do predictions.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.masking_utils import create_causal_mask

torch.set_grad_enabled(False)  # prediction only, no training
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
gpt2 = model.transformer  # word lookup + position lookup + 12 blocks + final cleanup step
NUM_BLOCKS = len(gpt2.h)

print("Device:", device)
print("Number of blocks:", NUM_BLOCKS)

/Users/arinpatel/Desktop/BTP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1022 23:45:33.058000 50143 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7881.63it/s]

Device: cpu
Number of blocks: 12


## Step 2: Normal run (no changes)
This is the reference answer. Everything later is compared against it.

In [3]:
PROMPT = "I live in Paris, so I speak"
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)
print("Word-pieces:", [tokenizer.decode(t) for t in input_ids[0]])

def top_predictions(logits, k=5):
    """Top k next words and their chances."""
    probs = logits[0, -1].softmax(-1)
    values, indices = probs.topk(k)
    return [(tokenizer.decode(i), v.item()) for i, v in zip(indices, values)]

normal_logits = model(input_ids, use_cache=False).logits
print("\nNormal model - top 5 next words:")
for word, p in top_predictions(normal_logits):
    print(f"  {word!r:12} {p:6.2%}")

Word-pieces: ['I', ' live', ' in', ' Paris', ',', ' so', ' I', ' speak']



Normal model - top 5 next words:
  ' French'    63.05%
  ' English'    4.93%
  ' to'         3.16%
  ' a'          2.94%
  ' with'       1.76%


## Step 3: Run the model one block at a time
These small functions do the same work as the model, but split up so we can stop between blocks:
- `embed` turns word-pieces into the starting table of numbers.
- `run_blocks(state, start, end)` runs blocks `start` to `end-1`.
- `run_until(ids, layer)` runs from the beginning up to and including `layer`, then **pauses**.
- `resume_from(state, layer)` **resumes** from block `layer + 1` and finishes the prediction.

In [4]:
def embed(input_ids):
    positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
    return gpt2.wte(input_ids) + gpt2.wpe(positions)  # word lookup + position lookup

def run_blocks(state, start, end):
    """Run blocks start, start+1, ..., end-1."""
    positions = torch.arange(state.shape[1], device=state.device).unsqueeze(0)
    # rule that each word can only look at earlier words (same as the normal model)
    mask = create_causal_mask(config=model.config, inputs_embeds=state, attention_mask=None,
                              past_key_values=None, position_ids=positions)
    for i in range(start, end):
        state = gpt2.h[i](state, attention_mask=mask, position_ids=positions)
    return state

def final_logits(state):
    return model.lm_head(gpt2.ln_f(state))  # final cleanup + turn numbers into word scores

def run_until(input_ids, layer):
    """Run from the start through block `layer`, then PAUSE and return the state."""
    return run_blocks(embed(input_ids), 0, layer + 1)

def resume_from(state, layer):
    """RESUME from block `layer + 1` to the end using a saved state."""
    return final_logits(run_blocks(state, layer + 1, NUM_BLOCKS))

## Step 4: A counter that records which blocks actually run
Every time a block runs, its number is added to `blocks_run`. We use this to **prove** earlier blocks are not rerun after resuming.

In [5]:
for h in globals().get("counter_hooks", []):  # safe to rerun this cell
    h.remove()

blocks_run = []
counter_hooks = [
    block.register_forward_hook(lambda module, inputs, output, i=i: blocks_run.append(i))
    for i, block in enumerate(gpt2.h)
]
print("Counter attached to", len(counter_hooks), "blocks")

Counter attached to 12 blocks


## Step 5: Check that the block-by-block run matches the normal model
We must confirm this before changing anything.

In [6]:
blocks_run.clear()
manual_logits = final_logits(run_blocks(embed(input_ids), 0, NUM_BLOCKS))

diff = (manual_logits - normal_logits).abs().max().item()
print("Blocks run:", blocks_run)
print("Largest difference from normal model:", diff)
print("Same top 5 words?", [w for w, _ in top_predictions(manual_logits)] == [w for w, _ in top_predictions(normal_logits)])
assert diff < 1e-4, "Block-by-block run does not match the normal model"
print("OK: block-by-block run matches the normal model")

Blocks run: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Largest difference from normal model: 0.0
Same top 5 words? True
OK: block-by-block run matches the normal model


## Step 6: PAUSE after a chosen block and SAVE its output
Blocks are numbered 0 to 11. We pause after block 5 (the 6th block).

In [7]:
PAUSE_LAYER = 5

blocks_run.clear()
paused_state = run_until(input_ids, PAUSE_LAYER)  # <-- PAUSE happens here
saved_state = paused_state.clone()                 # <-- SAVE a separate copy

print(f"PAUSED after block {PAUSE_LAYER}")
print("Blocks run so far:", blocks_run)
print("Saved output size:", tuple(saved_state.shape), " (1 sentence, 8 word-pieces, 768 numbers each)")
print("First 5 numbers for ' Paris':", saved_state[0, 3, :5].tolist())

PAUSED after block 5
Blocks run so far: [0, 1, 2, 3, 4, 5]
Saved output size: (1, 8, 768)  (1 sentence, 8 word-pieces, 768 numbers each)
First 5 numbers for ' Paris': [-5.1834259033203125, 5.828190803527832, 2.0465593338012695, 3.2916336059570312, -4.058125019073486]


## Step 7: RESUME with the saved output unchanged (check)
If pausing and resuming is done correctly, we should get **exactly** the normal answer, and only blocks 6 to 11 should run.

In [8]:
blocks_run.clear()
resumed_logits = resume_from(saved_state.clone(), PAUSE_LAYER)  # <-- RESUME at block PAUSE_LAYER + 1

print(f"RESUMED at block {PAUSE_LAYER + 1}. Blocks run:", blocks_run)
print("Largest difference from normal model:", (resumed_logits - normal_logits).abs().max().item())
assert blocks_run == list(range(PAUSE_LAYER + 1, NUM_BLOCKS)), "Earlier blocks were rerun!"
assert torch.allclose(resumed_logits, normal_logits, atol=1e-4)
print("OK: resuming gives the normal answer and earlier blocks were not rerun")

RESUMED at block 6. Blocks run: [6, 7, 8, 9, 10, 11]
Largest difference from normal model: 0.0
OK: resuming gives the normal answer and earlier blocks were not rerun


## Step 8: MODIFY the saved output
We take the saved numbers for the word **' Paris'** and replace them with the numbers block 5 produces for **' Berlin'**
in the sentence *"I live in Berlin, so I speak"*. Nothing else in the saved output is touched.

In [9]:
OTHER_PROMPT = "I live in Berlin, so I speak"
other_ids = tokenizer(OTHER_PROMPT, return_tensors="pt").input_ids.to(device)
assert other_ids.shape == input_ids.shape, "Both sentences must have the same number of word-pieces"

CITY_POS = 3  # position of ' Paris' / ' Berlin'
print("Replacing:", repr(tokenizer.decode(input_ids[0, CITY_POS])), "->", repr(tokenizer.decode(other_ids[0, CITY_POS])))

other_state = run_until(other_ids, PAUSE_LAYER)  # block 5 output for the Berlin sentence

modified_state = saved_state.clone()
modified_state[0, CITY_POS] = other_state[0, CITY_POS]  # <-- MODIFY happens here

changed = (modified_state != saved_state).any(-1)[0].nonzero().flatten().tolist()
print("Word-pieces whose numbers changed:", [tokenizer.decode(input_ids[0, i]) for i in changed])

Replacing: ' Paris' -> ' Berlin'
Word-pieces whose numbers changed: [' Paris']


## Step 9: RESUME with the modified output and compare

In [10]:
blocks_run.clear()
modified_logits = resume_from(modified_state, PAUSE_LAYER)  # <-- RESUME at block PAUSE_LAYER + 1
print(f"RESUMED at block {PAUSE_LAYER + 1}. Blocks run:", blocks_run)
assert blocks_run == list(range(PAUSE_LAYER + 1, NUM_BLOCKS)), "Earlier blocks were rerun!"

print(f"\n{'Normal model':<26}{'Modified model':<26}")
for (w1, p1), (w2, p2) in zip(top_predictions(normal_logits), top_predictions(modified_logits)):
    print(f"  {w1!r:12} {p1:6.2%}      {w2!r:12} {p2:6.2%}")

RESUMED at block 6. Blocks run: [6, 7, 8, 9, 10, 11]

Normal model              Modified model            
  ' French'    63.05%      ' German'    57.68%
  ' English'    4.93%      ' to'         5.95%
  ' to'         3.16%      ' English'    5.07%
  ' a'          2.94%      ' a'          3.49%
  ' with'       1.76%      ' the'        2.05%


## Step 10: Let both models write a few more words
Each new word needs a fresh pass through the model (we are not using the model's memory feature).
For the modified model, every pass pauses after block 5, swaps in the ' Berlin' numbers, and resumes.
The swapped numbers stay valid because the word ' Paris' only depends on the words before it, which never change.

In [11]:
def swap_city(state):
    state = state.clone()
    state[0, CITY_POS] = other_state[0, CITY_POS]
    return state

def continue_text(ids, n_words, modify=None):
    for _ in range(n_words):
        if modify is None:
            logits = model(ids, use_cache=False).logits
        else:
            state = modify(run_until(ids, PAUSE_LAYER))  # pause + modify
            logits = resume_from(state, PAUSE_LAYER)     # resume
        next_id = logits[0, -1].argmax().view(1, 1)      # always pick the top word
        ids = torch.cat([ids, next_id], dim=1)
    return tokenizer.decode(ids[0])

print("Normal:  ", repr(continue_text(input_ids, 10)))
print("Modified:", repr(continue_text(input_ids, 10, modify=swap_city)))

Normal:   "I live in Paris, so I speak French. I'm not a French person. I"


Modified: "I live in Paris, so I speak German. I'm not a German speaker, but"


## Step 11: Does the pause point matter?
We repeat the same swap, pausing after each block in turn, and record the chance of ' French' and ' German'.

In [12]:
FRENCH, GERMAN = tokenizer.encode(" French")[0], tokenizer.encode(" German")[0]

print(f"{'Pause after block':<20}{'Top word':<12}{'French':>8}{'German':>9}")
for layer in range(NUM_BLOCKS):
    state = run_until(input_ids, layer)
    state[0, CITY_POS] = run_until(other_ids, layer)[0, CITY_POS]
    probs = resume_from(state, layer)[0, -1].softmax(-1)
    print(f"{layer:<20}{tokenizer.decode(probs.argmax())!r:<12}{probs[FRENCH].item():>8.1%}{probs[GERMAN].item():>9.1%}")

Pause after block   Top word      French   German
0                   ' German'       0.9%    62.2%
1                   ' German'       0.9%    61.8%
2                   ' German'       0.9%    59.0%
3                   ' German'       0.9%    58.7%
4                   ' German'       0.9%    58.5%


5                   ' German'       0.9%    57.7%
6                   ' German'       0.9%    57.4%
7                   ' German'       0.9%    57.3%
8                   ' German'       9.3%    44.1%
9                   ' French'      24.6%    22.8%


10                  ' French'      33.9%    12.0%
11                  ' French'      63.0%     0.6%
